In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

from sklearn.linear_model import LinearRegression

In [3]:
df = pd.read_csv("Dataset 2.csv")

print(df.head())

   UserID  Age  Gender SubscriptionType  WatchHoursPerWeek  DevicesUsed  \
0    1001   22  Female            Basic                 23            5   
1    1002   55    Male            Basic                  9            4   
2    1003   49    Male            Basic                  8            3   
3    1004   39  Female          Premium                 19            5   
4    1005   38  Female          Premium                 23            5   

  FavoriteGenre  AdClicks  MonthlySpend SubscriptionRenewed  
0        Comedy        13           353                  No  
1         Drama        14           317                 Yes  
2        Comedy        16           309                  No  
3         Drama        45           833                 Yes  
4        Sci-Fi        24           804                 Yes  


In [4]:
print("Rows and Columns:", df.shape)

Rows and Columns: (750, 10)


In [5]:
print(df.columns)

Index(['UserID', 'Age', 'Gender', 'SubscriptionType', 'WatchHoursPerWeek',
       'DevicesUsed', 'FavoriteGenre', 'AdClicks', 'MonthlySpend',
       'SubscriptionRenewed'],
      dtype='object')


In [6]:
numerical_features = df.select_dtypes(include=['int64','float64']).columns
categorical_features = df.select_dtypes(include=['object']).columns

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
Index(['UserID', 'Age', 'WatchHoursPerWeek', 'DevicesUsed', 'AdClicks',
       'MonthlySpend'],
      dtype='object')

Categorical Features:
Index(['Gender', 'SubscriptionType', 'FavoriteGenre', 'SubscriptionRenewed'], dtype='object')


In [7]:
print("Total Missing Values:", df.isnull().sum().sum())

Total Missing Values: 0


In [8]:
avg_age = df["Age"].mean()
print("Average Age:", avg_age)

Average Age: 41.824


In [9]:
avg_watch = df["WatchHoursPerWeek"].mean()
print("Average Watch Hours:", avg_watch)

Average Watch Hours: 14.236


In [10]:
avg_spend = df["MonthlySpend"].mean()
print("Average Monthly Spend:", avg_spend)

Average Monthly Spend: 689.9053333333334


In [11]:
print(df["SubscriptionType"].value_counts())

SubscriptionType
Basic      342
Premium    279
VIP        129
Name: count, dtype: int64


In [12]:
renewed_percentage = (
    (df["SubscriptionRenewed"] == "Yes").mean()
) * 100

print("Renewal Percentage:", renewed_percentage)

Renewal Percentage: 46.266666666666666


In [13]:
df_encoded = df.copy()

label_encoders = {}

for col in df_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print(df_encoded.head())

   UserID  Age  Gender  SubscriptionType  WatchHoursPerWeek  DevicesUsed  \
0    1001   22       0                 0                 23            5   
1    1002   55       1                 0                  9            4   
2    1003   49       1                 0                  8            3   
3    1004   39       0                 1                 19            5   
4    1005   38       0                 1                 23            5   

   FavoriteGenre  AdClicks  MonthlySpend  SubscriptionRenewed  
0              1        13           353                    0  
1              2        14           317                    1  
2              1        16           309                    0  
3              2        45           833                    1  
4              5        24           804                    1  


In [14]:
X = df_encoded.drop(
    ["SubscriptionRenewed", "UserID"],
    axis=1
)

y = df_encoded["SubscriptionRenewed"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [17]:
y_pred_dt = dt_model.predict(X_test)

dt_accuracy = accuracy_score(
    y_test,
    y_pred_dt
)

print("Decision Tree Accuracy:", dt_accuracy)

Decision Tree Accuracy: 0.56


In [18]:
cm = confusion_matrix(
    y_test,
    y_pred_dt
)

print(cm)

[[48 34]
 [32 36]]


In [19]:
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

In [20]:
knn_accuracy = accuracy_score(
    y_test,
    y_pred_knn
)

print("Decision Tree Accuracy:", dt_accuracy)
print("KNN Accuracy:", knn_accuracy)

Decision Tree Accuracy: 0.56
KNN Accuracy: 0.6266666666666667


In [21]:
if knn_accuracy > dt_accuracy:
    print("KNN performs better.")
else:
    print("Decision Tree performs better.")

KNN performs better.


In [22]:
X_reg = df_encoded.drop(
    ["MonthlySpend", "UserID"],
    axis=1
)

y_reg = df_encoded["MonthlySpend"]

In [23]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [24]:
lr_model = LinearRegression()

lr_model.fit(
    X_train_reg,
    y_train_reg
)

LinearRegression()

In [25]:
new_user = [[
    30,     # Age
    1,      # Gender
    2,      # SubscriptionType
    25,     # WatchHoursPerWeek
    3,      # DevicesUsed
    1,      # FavoriteGenre
    15,     # AdClicks
    1       # SubscriptionRenewed
]]

In [26]:
predicted_spend = lr_model.predict(new_user)

print(
    "Predicted Monthly Spending:",
    predicted_spend[0]
)

Predicted Monthly Spending: 1365.636945855106


C:\Users\bhavy\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [27]:
print(dt_model.feature_importances_)

[0.19270241 0.04712665 0.0156751  0.17003478 0.07083496 0.08544427
 0.19928675 0.21889508]


In [ ]:
2. Why is subscription renewal a classification problem?

Because the outcome belongs to one of two categories:

Yes (Renewed)
No (Not Renewed)

The target variable is categorical, making it a classification task.

In [ ]:
3. Why is monthly spending a regression problem?

Monthly spending is a continuous numerical value (₹).

In [ ]:
4. Which algorithm performed better for renewal prediction?

Compare the accuracies:

In [28]:
if dt_accuracy > knn_accuracy:
    print("Decision Tree performs better.")
elif knn_accuracy > dt_accuracy:
    print("KNN performs better.")
else:
    print("Both models have the same accuracy.")

KNN performs better.


In [ ]:
5. How could Netflix use these predictions to improve customer retention?

Netflix can:

Identify users likely to cancel subscriptions.
Offer personalized discounts.
Recommend content based on viewing history.
Upgrade users to suitable subscription plans.
Send targeted engagement campaigns.
Improve customer experience for at-risk users.

These actions can increase renewal rates and overall revenue.